# Primer paso cargar el API Key y modelos

En este primer paso cargamos las librerías necesarias para poder trabajar sobre nuestros modelos y el PDF. Utilizaremos los modelos de OPENAI cargando la API KEY desde el archivo de configuración.

In [10]:
import os, getpass
from dotenv import load_dotenv
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

dotenv_loaded = load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("No se encontraron las API keys requeridas. Por favor, agrégalas al archivo .env o introdúcelas manualmente.")

GENERATION_MODEL = "gpt-5-mini"
EMBEDDING_MODEL = "text-embedding-ada-002"
PDF_PATH = Path("data/TENERIFE.pdf")

llm = init_chat_model(GENERATION_MODEL, model_provider="openai")
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)


# Recoger las secciones del documento

Aquí se esta cargando todo el documento para posteriormente coger todas las páginas y unificarlas en una sola página. De esta forma, podemos recorrer toda la página y coger la información por las diferentes secciones del documento sitios que ver y sitios para comer. También tengo que coger de que zona son los sitios para ver si se pregunta por norte o sur. Por último coger los lugarés de interés así que hago documentos separados por los lugarés de interés identificados por sección y zona.

In [35]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF en {PDF_PATH.resolve()}")

reader = PdfReader(str(PDF_PATH))

document = "".join(
    [page.extract_text().strip() for page in reader.pages if page.extract_text()]
)

visited_places = [
    "• Santa Cruz de Tenerife",
    "• San Cristóbal de La Laguna",
    "• Parque rural de Anaga",
    "• La Orotava",
    "• Puerto de La Cruz",
    "• Isla Baja (Icod de los Vinos, Garachico y Punta de Teno)",
    "• El Teide",
    "• Costa Adeje",
    "• Los Gigantes",
    "• Masca",
    "• El Médano / La Tejita",
    "• Los Cristianos",
]

other_places = [
    "• Sitios para comer (sobre todo os podré decir de zona norte)"
]


north_zone = [
    "• Santa Cruz de Tenerife",
    "• San Cristóbal de La Laguna",
    "• Parque rural de Anaga",
    "• La Orotava",
    "• Puerto de La Cruz",
    "• Isla Baja (Icod de los Vinos, Garachico y Punta de Teno)",
    "• El Teide"
]

south_zone = [
    "• Costa Adeje",
    "• Los Gigantes",
    "• Masca",
    "• El Médano / La Tejita",
    "• Los Cristianos"
]

places = [
    "• Santa Cruz de Tenerife",
    "• San Cristóbal de La Laguna",
    "• Parque rural de Anaga",
    "• La Orotava",
    "• Puerto de La Cruz",
    "• Isla Baja (Icod de los Vinos, Garachico y Punta de Teno)",
    "• El Teide",
    "• Costa Adeje",
    "• Los Gigantes",
    "• Masca",
    "• El Médano / La Tejita",
    "• Los Cristianos",
    "• Sitios para comer (sobre todo os podré decir de zona norte)"
]

def split_places(text: str, splits: list) -> dict:
    places = {}
    other_text = text
    
    for i in range(len(splits)):
        actual = splits[i]
        next = splits[i + 1] if i + 1 < len(splits) else None
        
        init_idx = other_text.find(actual)
        if init_idx == -1:
            continue
            
        if next:
            find_idx = other_text.find(next, init_idx + len(actual))
            if find_idx != -1:
                section = other_text[init_idx:find_idx]
            else:
                section = other_text[init_idx:]
        else:
            section = other_text[init_idx:]
            
        places[actual] = section.strip()
        
    return places

places_dict = split_places(document, places)

docs = [] 

zones = [
    "ZONA NORTE", "ZONA SUR"
]

for place in places_dict.keys():
    for page_number, page in enumerate(reader.pages):
        if place in page.extract_text():
            section = "SITIOS QUE VER" if place in visited_places else "SITIOS PARA COMER" if place in other_places else place
            zone = "ZONA NORTE" if place in north_zone else "ZONA SUR" if place in south_zone else None
            doc = Document(
                page_content=f"[Sección: {section} | Zona: {zone} | Lugar: {place}]\n{places_dict[place]}",
                metadata={
                    "section": section,
                    "zone": zone,
                    "page": page_number,
                    "page_label": reader.page_labels[page_number],
                    "total_pages": len(reader.pages),
                    "source": "TENERIFE.pdf"
                }
            )
            docs.append(doc)

docs


[Document(metadata={'section': 'SITIOS QUE VER', 'zone': 'ZONA NORTE', 'page': 0, 'page_label': '1', 'total_pages': 25, 'source': 'TENERIFE.pdf'}, page_content='[Sección: SITIOS QUE VER | Zona: ZONA NORTE | Lugar: • Santa Cruz de Tenerife]\n• Santa Cruz de Tenerife: \nSanta Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa \nCruz sería: \n- Aparcar en el aparcamiento del Parque Marítimo (ubicación). \n- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). \n- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). \n- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo \ndirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria - \nubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del \nPríncipe - ubicación). \n- Volver de nuevo a por el coche. \n- Ir hacia la playa de Las Teresitas (ubicación). \nLos puntos de interés sugeridos son estos: \no Parque Marí

# Chunking
Utizamos una estrategia de chunking cortando el texto cada 500 caracteres primero intenta cortar por párrafos \n\n, si sigue siendo muy largo corta por saltos de línea \n, luego por espacios, y como último recurso por caracteres individuales. Para el próximo recorte va 80 caracteres más atrás y por si había recortado la información y había perdido contexto.

In [37]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    add_start_index=True,
)

chunks = text_splitter.split_documents(docs)

for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i
    doc.metadata["source_name"] = PDF_PATH.name

print("Número de chunks:", len(chunks))
print("\nEjemplo de chunk:\n")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Número de chunks: 45

Ejemplo de chunk:

[Sección: SITIOS QUE VER | Zona: ZONA NORTE | Lugar: • Santa Cruz de Tenerife]
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación).

Metadata: {'section': 'SITIOS QUE VER', 'zone': 'ZONA NORTE', 'page': 0, 'page_label': '1', 'total_pages': 25, 'source': 'TENERIFE.pdf', 'start_index': 0, 'chunk_id': 0, 'source_name': 'TENERIFE.pdf'}


# Embeddings
Utilizaremos InMemoryVectorStore para guardar en memoria nuestros vectores para esta práctica. Con esto lo que hacemos es indexar los chunks y es más fácil para el LLM encontrar ese chunk que podría contener información relevante para poder contestar a lo que se le pregunta.

In [38]:
vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(chunks)

print("Documentos indexados:", len(document_ids))
print("Primeros IDs:", document_ids[:3])

Documentos indexados: 45
Primeros IDs: ['2eccc757-eb8a-472b-8769-03834f5eccaa', 'fecda431-125b-489a-97af-4bc7863230d1', 'ee8677af-e186-4ce7-b241-f64d08c10349']


# Respuesta del modelo utilizando el RAG
Se le especifica al modelo que es un guía turístico de Tenerife y que tiene que dar las respuestas en español y no inventarse nada que no encuentre en los documentos. También tiene que citar de donde sale la iformación.

In [41]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Eres un guía turístico experto en la isla de Tenerife. "
            "Responde en español usando únicamente el contexto proporcionado. "
            "Si el contexto no contiene la respuesta, di claramente: 'No tengo información suficiente en la base documental'. "
            "No inventes datos. Cita las fuentes usadas al final.",
        ),
        (
            "user",
            "Pregunta:\n{question}\n\nContexto recuperado:\n{context}\n\nRespuesta:",
        ),
    ]
)

In [42]:
def format_docs_with_sources(documentos):
    bloques = []
    for i, doc in enumerate(documentos, start=1):
        source = doc.metadata.get("source_name", "fuente desconocida")
        page = doc.metadata.get("page", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        bloques.append(
            f"[Fuente {i}: {source}, página {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(bloques)


def answer_with_rag(question: str, k: int = 3, show_context: bool = False):
    retrieved_docs = vector_store.similarity_search(question, k=k)
    context = format_docs_with_sources(retrieved_docs)

    if show_context:
        print("CONTEXTO RECUPERADO")
        print(context)
        print("=" * 80)

    messages = rag_prompt.invoke({"question": question, "context": context})
    answer = llm.invoke(messages)

    return {
        "question": question,
        "answer": answer.content,
        "context": retrieved_docs,
        "prompt": messages,
    }

result = answer_with_rag(
    "¿Que sitios ver en la zona sur?",
    k=3,
    show_context=True,
)

print(result["answer"])

CONTEXTO RECUPERADO
[Fuente 1: TENERIFE.pdf, página 22, chunk 36]
[Sección: SITIOS QUE VER | Zona: ZONA SUR | Lugar: • Los Cristianos]
• Los Cristianos 
De la zona de Los Cristianos que me venga a la mente os recomendaría la Playa de 
Los Cristianos o la Playa de Las Vistas y creo que poco más (la imagen es la Playa de 
Las Vistas).OTRAS VAINAS

[Fuente 2: TENERIFE.pdf, página 15, chunk 25]
[Sección: SITIOS QUE VER | Zona: ZONA SUR | Lugar: • Costa Adeje]
• Costa Adeje 
Costa Adeje es actualmente la zona turística por excelencia de Tenerife. Es donde se 
concentran la mayor cantidad de hoteles de lujo, las zonas de playa de arena blanca 
(en el norte de la isla las playas son más de arena negra), las zonas de ocio (pubs, 
discotecas, parques acuáticos, etc.). 
De esta zona os recomendaríamos las siguientes playas: 
o Playa de Torviscaso Playa de Troya

[Fuente 3: TENERIFE.pdf, página 11, chunk 16]
[Sección: SITIOS QUE VER | Zona: ZONA NORTE | Lugar: • Isla Baja (Icod de los Vinos, Gara